<a href="https://colab.research.google.com/github/bushracpu123/assignment3/blob/main/Fine_Tuning_Qwen2_Audio_Urdu_Hematology_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Qwen2-Audio-7B for Urdu Hematology Audio Question Answering
### End-to-End Multilingual Audio Language Model (ALM) on Blood Cell Cytology
**Target Task:** Raw Urdu Spoken Audio Question (`.wav`) ➔ Model Auditory Processing ➔ Urdu Clinical Answer Generation (`answer_ur`)  
**Base Architecture:** `Qwen/Qwen2-Audio-7B-Instruct`  
**Optimization Method:** 4-bit NF4 Quantization (BitsAndBytes) + Parameter-Efficient LoRA (PEFT)  
**Target Device:** Google Colab T4 GPU (16 GB VRAM)

---
### Why Qwen2-Audio-7B is Selected:
1. **Direct Audio Input:** It processes raw audio waveforms natively via an integrated convolutional audio encoder.
2. **Native Urdu Multilingual Tokenizer:** The base LLM features a 152,000-token multilingual vocabulary that natively tokens and outputs fluent Urdu script without transliteration.
3. **Medical Reasoning:** Unlike standard ASR models that only transcribe words, this model performs clinical hematological reasoning to answer diagnostic questions about blood cell morphology.


## Step 1: Install Dependencies
Installs Hugging Face Transformers (v4.45+), PEFT, BitsAndBytes for 4-bit quantization, Accelerate, Librosa, and Soundfile for 16 kHz audio processing.

In [1]:
import subprocess, sys

packages = [
    "transformers>=4.45.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.33.0",
    "librosa",
    "soundfile",
    "pandas",
    "tqdm"
]

print("Installing required packages for Qwen2-Audio fine-tuning...")
for p in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

print("Installation complete.")


Installing required packages for Qwen2-Audio fine-tuning...
Installation complete.


## Step 2: Mount Google Drive
Connects Google Drive where your dataset (`speech_hemavqa_master_qa.csv`) and Urdu audios are stored.

In [2]:
from google.colab import drive
import os

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")
else:
    print("Google Drive is already mounted.")


Mounted at /content/drive
Google Drive mounted successfully.


## Step 3: Locate Files & Unpack Urdu Audio Dataset
Searches Google Drive for `speech_hemavqa_master_qa.csv` and the `Audio urdu` folder. If any `.zip` archives are present in `Audio urdu`, they are automatically unzipped into `/content/urdu_audio_data`.

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, glob
root = "/content/drive/MyDrive/hematology/hematology dataset"
print("Items in dataset folder:", os.listdir(root))

for z in glob.glob(root + "/**/audios_ur_*.zip", recursive=True)[:10]:
    print("ZIP:", z)

Mounted at /content/drive
Items in dataset folder: ['test_urdu.json', 'images.zip', 'Readme.md', 'LICENSE.txt', 'train.json', 'test_english.json', 'requirements.txt', 'train', 'PBC_dataset_normal_DIB', 'test', 'annotations', 'urdu_translation_cache.json', 'Audio urdu', 'Speech_HemaVQA_outputs', 'speech_hemavqa_master_qa.csv', 'Audio urdu (1)']
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1_1000.zip
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1001_2000.zip
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_2001_3000.zip
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_3001_4000.zip
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_4001_5000.zip
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_5001_6000.zip
ZIP: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_6001_7000.zip
ZIP: /

In [3]:
import os, glob, zipfile, pandas as pd, re

print("Locating dataset directory and Urdu audio files in Google Drive...")

# 1. Search for master CSV
csv_candidates = [
    "/content/drive/MyDrive/hematology/hematology dataset/speech_hemavqa_master_qa.csv",
    "/content/drive/MyDrive/hematology (1)/hematology dataset/speech_hemavqa_master_qa.csv",
    "/content/drive/MyDrive/hematology/speech_hemavqa_master_qa.csv",
]
CSV_PATH = None
for cand in csv_candidates:
    if os.path.exists(cand):
        CSV_PATH = cand
        break

if not CSV_PATH:
    hits = glob.glob("/content/drive/MyDrive/**/speech_hemavqa_master_qa.csv", recursive=True)
    if hits:
        CSV_PATH = hits[0]

if not CSV_PATH:
    raise FileNotFoundError("Could not find speech_hemavqa_master_qa.csv in Google Drive. Please ensure your Drive is mounted.")

print(f"Master CSV found: {CSV_PATH}")
BASE_DIR = os.path.dirname(CSV_PATH)
print(f"Base Directory: {BASE_DIR}")

# Load metadata
df_all = pd.read_csv(CSV_PATH).dropna(subset=['row', 'question_ur', 'answer_ur'])
df_all['row'] = df_all['row'].astype(int)
print(f"Loaded Master CSV: {len(df_all):,} records.")

# 2. Local SSD extraction directory for high-speed training
LOCAL_URDU_AUDIO = "/content/urdu_audio_data"
os.makedirs(LOCAL_URDU_AUDIO, exist_ok=True)

# 3. Locate Urdu audio archives or folders in Drive
print("Scanning for Urdu audio archives in Google Drive...")
search_dirs = [
    os.path.join(BASE_DIR, "Audio urdu"),
    os.path.join(BASE_DIR, "Audio Dataset"),
    os.path.join(BASE_DIR, "original urdu audio WAV"),
    BASE_DIR
]

# Find any audio zip files
urdu_zips = []
for d in search_dirs:
    if os.path.exists(d):
        urdu_zips.extend(glob.glob(f"{d}/**/audios_ur_*.zip", recursive=True))
        urdu_zips.extend(glob.glob(f"{d}/**/audios_*.zip", recursive=True))

# Remove duplicates while preserving order
urdu_zips = list(dict.fromkeys(urdu_zips))
print(f"Found {len(urdu_zips)} audio zip archive(s):")
for z in urdu_zips[:5]:
    print("  ", z)

# Unpack audio files if not already unpacked
existing_wavs = glob.glob(f"{LOCAL_URDU_AUDIO}/**/*.wav", recursive=True)
if len(existing_wavs) < 100 and urdu_zips:
    for azip in urdu_zips:
        print(f"Unpacking: {azip} ...")
        try:
            with zipfile.ZipFile(azip, 'r') as z:
                z.extractall(LOCAL_URDU_AUDIO)
        except Exception as e:
            print(f"Error extracting {azip}: {e}")
        # Stop once we have sufficient WAV files for training
        if len(glob.glob(f"{LOCAL_URDU_AUDIO}/**/*.wav", recursive=True)) >= 1500:
            break

# Also check for direct unzipped WAV files in Audio urdu
for d in search_dirs:
    if os.path.exists(d):
        direct_wavs = glob.glob(f"{d}/**/*.wav", recursive=True)
        if len(direct_wavs) > 0 and len(glob.glob(f"{LOCAL_URDU_AUDIO}/**/*.wav", recursive=True)) < 100:
            print(f"Copying direct WAV files from {d} ...")
            import shutil
            for w in direct_wavs[:1500]:
                shutil.copy(w, LOCAL_URDU_AUDIO)
            break

all_local_wavs = glob.glob(f"{LOCAL_URDU_AUDIO}/**/*.wav", recursive=True)
print(f"\nTotal local audio files available: {len(all_local_wavs)}")
if all_local_wavs:
    print(f"Sample filename: {os.path.basename(all_local_wavs[0])}")
else:
    raise FileNotFoundError("No audio .wav files found! Please verify that your 'Audio urdu' folder contains WAV files or audio zip archives.")


Locating dataset directory and Urdu audio files in Google Drive...
Master CSV found: /content/drive/MyDrive/hematology/hematology dataset/speech_hemavqa_master_qa.csv
Base Directory: /content/drive/MyDrive/hematology/hematology dataset
Loaded Master CSV: 107,911 records.
Scanning for Urdu audio archives in Google Drive...
Found 12 audio zip archive(s):
   /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1_1000.zip
   /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1001_2000.zip
   /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_2001_3000.zip
   /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_3001_4000.zip
   /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_4001_5000.zip
Unpacking: /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1_1000.zip ...
Error extracting /content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1_

FileNotFoundError: No audio .wav files found! Please verify that your 'Audio urdu' folder contains WAV files or audio zip archives.

## Step 4: Prepare 1,000 Training & Validation Pairs
Extracts the question audio files (`_ur_q` or `_q`), matches them with the corresponding row in `speech_hemavqa_master_qa.csv` by row number or filename, and splits them into 800 training and 200 validation samples.

In [6]:
import glob
print("WAVs extracted:", len(glob.glob("/content/urdu_audio_data/**/*.wav", recursive=True)))
print("ZIPs visible:", glob.glob("/content/drive/MyDrive/hematology/hematology dataset/**/audios_ur_*.zip", recursive=True)[:5])

WAVs extracted: 0
ZIPs visible: ['/content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1_1000.zip', '/content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_1001_2000.zip', '/content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_2001_3000.zip', '/content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_3001_4000.zip', '/content/drive/MyDrive/hematology/hematology dataset/Audio urdu/audios_ur_4001_5000.zip']


In [7]:
import zipfile, glob, os

AUDIO_ZIP_DIR = "/content/drive/MyDrive/hematology/hematology dataset/Audio urdu"
LOCAL_URDU_AUDIO = "/content/urdu_audio_data"
os.makedirs(LOCAL_URDU_AUDIO, exist_ok=True)

zips = sorted(glob.glob(AUDIO_ZIP_DIR + "/audios_ur_*.zip"))[:5]
for z in zips:
    print("Extracting:", os.path.basename(z))
    with zipfile.ZipFile(z) as zf:
        zf.extractall(LOCAL_URDU_AUDIO)

wavs = glob.glob(LOCAL_URDU_AUDIO + "/**/*.wav", recursive=True)
print("Total WAVs:", len(wavs))
print("Sample:", [os.path.basename(w) for w in wavs[:3]])

Extracting: audios_ur_1001_2000.zip
Extracting: audios_ur_18539_20000.zip
Extracting: audios_ur_1_1000.zip
Extracting: audios_ur_20001_30000 (1).zip
Extracting: audios_ur_20001_30000.zip
Total WAVs: 26888
Sample: ['row22400_id16533_img16533_ur_q_female.wav', 'row25165_id20288_img20288_ur_q_female.wav', 'row24462_id19385_img19385_ur_q_female.wav']


In [8]:
import os, re, glob, pandas as pd

LOCAL_URDU_AUDIO = "/content/urdu_audio_data"
all_local_wavs = glob.glob(f"{LOCAL_URDU_AUDIO}/**/*.wav", recursive=True)

# Urdu question audio only
question_wavs = [w for w in all_local_wavs if "_ur_q" in os.path.basename(w)]
print("Total WAVs:", len(all_local_wavs), "| Urdu question WAVs:", len(question_wavs))

meta = df_all.drop_duplicates("row").set_index("row")[["id", "question_ur", "answer_ur"]].to_dict("index")
pat = re.compile(r"row(\d+)_id(\d+)", re.IGNORECASE)

valid_samples, id_mismatch, no_row = [], 0, 0
for wav_path in question_wavs:
    m = pat.search(os.path.basename(wav_path))
    if not m:
        continue
    row_num, file_id = int(m.group(1)), int(m.group(2))
    rec = meta.get(row_num)
    if rec is None:
        no_row += 1
        continue
    if int(rec["id"]) != file_id:
        id_mismatch += 1
        continue
    q_ur, a_ur = str(rec["question_ur"]).strip(), str(rec["answer_ur"]).strip()
    if len(q_ur) > 3 and len(a_ur) > 3:
        valid_samples.append({"row": row_num, "audio_path": wav_path,
                              "question_ur": q_ur, "answer_ur": a_ur})

print(f"Matched: {len(valid_samples)} | id mismatch: {id_mismatch} | row not in CSV: {no_row}")

df_target = pd.DataFrame(valid_samples).sample(frac=1, random_state=42).reset_index(drop=True)
df_target = df_target.iloc[:1000]   # cap at 1000 samples; change if you want more

train_size = int(0.8 * len(df_target))
train_df = df_target.iloc[:train_size].reset_index(drop=True)
val_df = df_target.iloc[train_size:].reset_index(drop=True)
print(f"train {len(train_df)} | val {len(val_df)}")

train_dataset = UrduHematologyAudioDataset(train_df)
val_dataset = UrduHematologyAudioDataset(val_df)
print("Datasets ready:", len(train_dataset), len(val_dataset))

Total WAVs: 26888 | Urdu question WAVs: 13444
Matched: 147 | id mismatch: 13297 | row not in CSV: 0
train 117 | val 30


NameError: name 'UrduHematologyAudioDataset' is not defined

In [9]:
from IPython.display import Audio, display

bad = []
for w in question_wavs[:300]:
    m = pat.search(os.path.basename(w))
    r, i = int(m.group(1)), int(m.group(2))
    if int(meta[r]["id"]) != i:
        bad.append((w, r, i))

for w, r, i in bad[:3]:
    print("FILE:", os.path.basename(w))
    print(" CSV row", r, "-> id", meta[r]["id"], "|", meta[r]["question_ur"][:80])
    print(" CSV rows with id", i, ":")
    print(df_all[df_all["id"] == i][["row", "source_file", "question_ur"]].head(3).to_string())
    display(Audio(w))

FILE: row22400_id16533_img16533_ur_q_female.wav
 CSV row 22400 -> id 23400 | اس لیوکیمیا سیل میں سائٹوپلاسمیک ویکیولز کیا ہیں؟
 CSV rows with id 16533 :
         row source_file                                    question_ur
16207  16208       train  اس تصویر میں سائٹوپلاسما کی مورفولوجی کیا ہے؟


FILE: row25165_id20288_img20288_ur_q_female.wav
 CSV row 25165 -> id 26334 | اس سیل کا نیوکلیئر کرومٹین پیٹرن کیا ہے؟
 CSV rows with id 20288 :
         row source_file                                           question_ur
19563  19564       train  اس سیل میں سائٹوپلاسمیک ویکیولز کی مورفولوجی کیا ہے؟


FILE: row24462_id19385_img19385_ur_q_female.wav
 CSV row 24462 -> id 25602 | لیوکیمیا سیل کی اس تصویر میں کروماتین کی مورفولوجی کیا ہے؟
 CSV rows with id 19385 :
         row source_file                               question_ur
18739  18740       train  اس سیل کا نیوکلیئر کرومٹین پیٹرن کیا ہے؟


In [5]:
import re

print("Matching local Urdu question audios with master CSV metadata...")

# Filter question WAV files only (exclude answer audio files)
question_wavs = [w for w in all_local_wavs if "_q" in os.path.basename(w) or "question" in w.lower()]

if not question_wavs:
    # If no files have '_q' in name, use all WAVs
    question_wavs = all_local_wavs

print(f"Identified {len(question_wavs)} potential question audio files.")

# Build row-indexed lookup table from master CSV for fast matching
df_meta_by_row = df_all.set_index('row')

valid_samples = []

for wav_path in question_wavs:
    fname = os.path.basename(wav_path)

    # Method 1: Extract row number via regex from filename: row{ROW}_...
    row_num = None
    m = re.search(r'row(\d+)', fname, re.IGNORECASE)
    if m:
        row_num = int(m.group(1))

    # Method 2: Match by exact audio filename in master CSV
    if row_num is None or row_num not in df_meta_by_row.index:
        matched = df_all[df_all['audio_ur_q'] == fname]
        if not matched.empty:
            row_num = int(matched.iloc[0]['row'])

    # If matched with CSV record, add to cohort
    if row_num is not None and row_num in df_meta_by_row.index:
        meta_row = df_meta_by_row.loc[row_num]
        q_ur = str(meta_row['question_ur']).strip()
        a_ur = str(meta_row['answer_ur']).strip()

        # Verify valid non-empty Urdu text
        if len(q_ur) > 3 and len(a_ur) > 3:
            valid_samples.append({
                "row": row_num,
                "audio_path": wav_path,
                "question_ur": q_ur,
                "answer_ur": a_ur
            })
            if len(valid_samples) >= 1000:
                break

# Build final DataFrame
df_target = pd.DataFrame(valid_samples)

if len(df_target) == 0:
    raise ValueError(
        f"Could not match any of the {len(question_wavs)} audio files with rows in speech_hemavqa_master_qa.csv.\n"
        f"Sample audio filename was: {os.path.basename(question_wavs[0]) if question_wavs else 'None'}.\n"
        f"Please check your audio filenames or zip archives in Drive."
    )

print(f"\nSuccessfully matched {len(df_target)} Urdu Audio-QA pairs!")

# 80/20 Train/Validation Split
train_size = int(0.8 * len(df_target))
train_df = df_target.iloc[:train_size].reset_index(drop=True)
val_df = df_target.iloc[train_size:].reset_index(drop=True)

print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")

print("\nSample Record:")
print("Row:", train_df.loc[0, 'row'])
print("Audio File:", os.path.basename(train_df.loc[0, 'audio_path']))
print("Urdu Question:", train_df.loc[0, 'question_ur'])
print("Urdu Answer:", train_df.loc[0, 'answer_ur'])


Matching local Urdu question audios with master CSV metadata...
Identified 0 potential question audio files.


ValueError: Could not match any of the 0 audio files with rows in speech_hemavqa_master_qa.csv.
Sample audio filename was: None.
Please check your audio filenames or zip archives in Drive.

## Step 5: Load Qwen2-Audio-7B with 4-Bit Quantization
Loads the base `Qwen/Qwen2-Audio-7B-Instruct` model in 4-bit NF4 precision. This reduces VRAM from 28 GB to ~5.5 GB, allowing it to fit on a Colab T4 GPU.

In [10]:
import torch
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")

# 4-bit NF4 Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit precision...")
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)

# Configure LoRA for Attention Projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Device: CUDA
GPU: Tesla T4 (VRAM: 14.56 GB)
Loading Qwen/Qwen2-Audio-7B-Instruct in 4-bit precision...


config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

... (3 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

trainable params: 20,709,376 || all params: 8,417,804,288 || trainable%: 0.2460


## Step 6: Dataset Definition and Audio Processing
Reads raw audio at 16,000 Hz, constructs the conversation prompt with auditory placeholders, and masks user prompt tokens so the loss is calculated only on the clinical answer.

In [11]:
import librosa
from torch.utils.data import Dataset, DataLoader

URDU_SYSTEM_PROMPT = "آپ ہیماتولوجی کے ماہر ڈاکٹر ہیں۔ اس آڈیو میں پوچھے گئے سوال کا تفصیلی اور درست طبی جواب اردو میں دیں۔"

class UrduHematologyAudioDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row['audio_path']
        answer_text = row['answer_ur']

        # Load audio at 16 kHz mono
        audio, _ = librosa.load(audio_path, sr=16000)

        return {
            "audio": audio,
            "answer_text": answer_text,
            "row": row['row']
        }

def collate_fn(batch):
    audios = [item["audio"] for item in batch]
    answers = [item["answer_text"] for item in batch]

    # Format conversational prompt
    texts = []
    for ans in answers:
        conv = [
            {
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "dummy.wav"},
                    {"type": "text", "text": URDU_SYSTEM_PROMPT}
                ]
            },
            {"role": "assistant", "content": ans}
        ]
        prompt_text = processor.apply_chat_template(conv, add_generation_prompt=False, tokenize=False)
        texts.append(prompt_text)

    # Process batch using AutoProcessor
    inputs = processor(
        text=texts,
        audio=audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    # Labels: duplicate input_ids and mask padding tokens with -100
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    inputs["labels"] = labels
    return inputs

train_dataset = UrduHematologyAudioDataset(train_df)
val_dataset = UrduHematologyAudioDataset(val_df)

print(f"Datasets configured: Train={len(train_dataset)}, Val={len(val_dataset)}")


Datasets configured: Train=117, Val=30


## Step 7: LoRA Fine-Tuning Loop
Executes 3 training epochs with gradient accumulation (effective batch size: 8) using `paged_adamw_8bit` optimizer.

In [12]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
EPOCHS = 3
LEARNING_RATE = 2e-4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.05), num_training_steps=total_steps)

print(f"Starting Fine-Tuning across {EPOCHS} epochs ({total_steps} optimization steps)...")

OUTPUT_LORA_DIR = os.path.join(BASE_DIR, "Speech_HemaVQA_outputs", "best_qwen2audio_urdu_lora")
os.makedirs(OUTPUT_LORA_DIR, exist_ok=True)

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for step, batch in enumerate(pbar, 1):
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        total_train_loss += loss.item() * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix({"Loss": f"{total_train_loss / step:.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation evaluation
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for val_batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            val_batch = {k: v.to(device) for k, v in val_batch.items() if isinstance(v, torch.Tensor)}
            val_outputs = model(**val_batch)
            total_val_loss += val_outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print(f"\n--- Epoch {epoch} Results | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} ---")

    # Save checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"Saving best LoRA checkpoint to: {OUTPUT_LORA_DIR}")
        model.save_pretrained(OUTPUT_LORA_DIR)

print("\nFine-tuning completed successfully!")


Starting Fine-Tuning across 3 epochs (42 optimization steps)...


Epoch 1/3 [Train]:   0%|          | 0/117 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 1/3 [Val]:   0%|          | 0/30 [00:00<?, ?it/s]


--- Epoch 1 Results | Train Loss: 6.8892 | Val Loss: 4.2007 ---
Saving best LoRA checkpoint to: /content/drive/MyDrive/hematology/hematology dataset/Speech_HemaVQA_outputs/best_qwen2audio_urdu_lora


Epoch 2/3 [Train]:   0%|          | 0/117 [00:00<?, ?it/s]

Epoch 2/3 [Val]:   0%|          | 0/30 [00:00<?, ?it/s]


--- Epoch 2 Results | Train Loss: 3.7586 | Val Loss: 3.4863 ---
Saving best LoRA checkpoint to: /content/drive/MyDrive/hematology/hematology dataset/Speech_HemaVQA_outputs/best_qwen2audio_urdu_lora


Epoch 3/3 [Train]:   0%|          | 0/117 [00:00<?, ?it/s]

Epoch 3/3 [Val]:   0%|          | 0/30 [00:00<?, ?it/s]


--- Epoch 3 Results | Train Loss: 3.4507 | Val Loss: 3.4379 ---
Saving best LoRA checkpoint to: /content/drive/MyDrive/hematology/hematology dataset/Speech_HemaVQA_outputs/best_qwen2audio_urdu_lora

Fine-tuning completed successfully!


## Step 8: Qualitative Inference on Urdu Spoken Question
Feeds an unseen Urdu audio question directly to the fine-tuned model and compares the generated Urdu clinical diagnosis with the ground truth.

In [13]:
from IPython.display import display, Audio, HTML

def run_urdu_audio_inference(audio_path, ground_truth_q=None, ground_truth_a=None):
    """Runs end-to-end inference on a raw Urdu audio question."""
    model.eval()

    # 1. Load audio at 16 kHz
    raw_audio, sr = librosa.load(audio_path, sr=16000)

    # 2. Construct conversation prompt
    conv = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "input.wav"},
                {"type": "text", "text": URDU_SYSTEM_PROMPT}
            ]
        }
    ]
    prompt_text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)

    # 3. Process inputs
    inputs = processor(
        text=prompt_text,
        audio=raw_audio,
        sampling_rate=16000,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items() if isinstance(v, torch.Tensor)}

    # 4. Generate answer
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            repetition_penalty=1.1,
            do_sample=False
        )

    gen_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    predicted_urdu = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

    # 5. Display Results
    display(HTML(f"<h4>Input Urdu Spoken Question Audio: {os.path.basename(audio_path)}</h4>"))
    display(Audio(audio_path))

    gt_q_html = f"<p><b>Ground-Truth Urdu Question:</b> {ground_truth_q}</p>" if ground_truth_q else ""
    gt_a_html = f"<p><b>Ground-Truth Urdu Answer:</b> {ground_truth_a}</p>" if ground_truth_a else ""

    display(HTML(f"""
    <div style="background:#1a237e;color:white;padding:16px;border-radius:10px;font-family:Arial;margin-top:10px;">
        <h3 style="margin-top:0;">Urdu Audio Language Model Diagnosis</h3>
        {gt_q_html}
        <p style="font-size:16px;color:#81d4fa;"><b>Model Predicted Answer (اردو):</b><br>{predicted_urdu}</p>
        {gt_a_html}
    </div>
    """))

    return predicted_urdu

# Test on validation sample
test_sample = val_df.iloc[0]
print("Running inference on test sample...")
run_urdu_audio_inference(
    audio_path=test_sample['audio_path'],
    ground_truth_q=test_sample['question_ur'],
    ground_truth_a=test_sample['answer_ur']
)


Running inference on test sample...


'سائٹوپلاسمکارن گول ہے، جس کا بین 250 و 500 ملیم سے کم ہوا ہے.'